# Medical Framework — Unified Pipeline
Single `imblearn.Pipeline` driven by `RandomizedSearchCV` (200 draws). Each step is one of the custom transformers from the `.py` modules; the search picks the best combination.

In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

from loader import load_data

from clean_data import MedianImputer, KNNImputerWrapper, IterativeModelImputer
from tame_outlier import IsolationForestTamer
from normalization import RobustScalerNorm, ZScoreNormalizationNorm
from feature_selection import SelectKBestFilter, TreeBasedSelection
from balance import IdentitySampler, SMOTESampler, BorderlineSMOTESampler
from model_training import (
    LogisticRegressionEstimator, RandomForestEstimator,
    XGBoostEstimator, LightGBMEstimator,
)

## Load & split

In [2]:
path = './Final/data'
typeData = 'csv'
y_column = 'CVD.event'

X, Y, all_mappings, y_mappings = load_data(path=f'{path}.{typeData}', y_column=y_column)
X = X.astype('float32')

print(f'X shape : {X.shape}')
print(f'Classes : {pd.Series(Y).value_counts().to_dict()}')

x_train, x_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y,
)

X shape : (7433, 55)
Classes : {0: 6596, 1: 837}


## Build the pipeline
Six stages: imputation → outlier flags → normalization → feature selection → balancing → classifier. The starting values are placeholders — `RandomizedSearchCV` swaps each step out below.

In [3]:
pipeline = Pipeline(steps=[
    ('imputer',    MedianImputer()),
    ('tamer',      'passthrough'),
    ('normalizer', ZScoreNormalizationNorm()),
    ('selector',   SelectKBestFilter(k=20)),
    ('balancer',   IdentitySampler()),
    ('classifier', LogisticRegressionEstimator()),
], memory='./cache')
pipeline

,steps,"[('imputer', ...), ('tamer', ...), ...]"
,transform_input,None
,memory,'./cache'
,verbose,False
,k,20
,C,1.0
,max_iter,1000


## Search space
Each sub-dict pins one classifier and lists compatible step choices + hyper-parameters. `RandomizedSearchCV` samples 200 combinations from the cross-product of all sub-dicts.

In [4]:
common_imputers    = [MedianImputer(), KNNImputerWrapper(n_neighbors=5), IterativeModelImputer()]
common_tamers      = ['passthrough', IsolationForestTamer()]
common_normalizers = [ZScoreNormalizationNorm(), RobustScalerNorm()]
common_balancers   = [IdentitySampler(), SMOTESampler(k_neighbors=3), BorderlineSMOTESampler(k_neighbors=3)]

param_grid = [
    # Logistic Regression + SelectKBest (k is only valid on SelectKBestFilter)
    {
        'imputer':     common_imputers,
        'tamer':       common_tamers,
        'normalizer':  common_normalizers,
        'selector':    [SelectKBestFilter()],
        'selector__k': [10, 20, 30],
        'balancer':    common_balancers,
        'classifier':  [LogisticRegressionEstimator()],
        'classifier__C': [0.1, 1.0, 10.0],
    },
    # Logistic Regression + TreeBased selector (no k parameter)
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LogisticRegressionEstimator()],
        'classifier__C': [0.1, 1.0, 10.0],
    },
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [RandomForestEstimator()],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth':    [None, 10, 20],
    },
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [XGBoostEstimator()],
        'classifier__n_estimators':  [100, 200],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__max_depth':     [3, 5],
    },
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LightGBMEstimator()],
        'classifier__n_estimators':  [100, 200],
        'classifier__learning_rate': [0.05, 0.1],
    },
]

n_combos = sum(int(np.prod([len(v) for v in g.values()])) for g in param_grid)
print(f'Total candidate configurations (full grid): {n_combos}')

Total candidate configurations (full grid): 1728


## Fit the search

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=200,
    scoring='roc_auc',
    cv=cv,
    n_jobs=2,
    pre_dispatch='n_jobs',
    verbose=2,
    refit=True,
    random_state=42,
    return_train_score=False,
)

search.fit(x_train, y_train)

Fitting 5 folds for each of 200 candidates, totalling 1000 fits


/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

[CV] END balancer=SMOTESampler(), classifier=RandomForestEstimator(), classifier__max_depth=None, classifier__n_estimators=100, imputer=IterativeModelImputer(), normalizer=RobustScalerNorm(), selector=TreeBasedSelection(), tamer=IsolationForestTamer(); total time=   0.1s
[CV] END balancer=SMOTESampler(), classifier=RandomForestEstimator(), classifier__max_depth=None, classifier__n_estimators=100, imputer=IterativeModelImputer(), normalizer=RobustScalerNorm(), selector=TreeBasedSelection(), tamer=IsolationForestTamer(); total time=   0.1s
[CV] END balancer=SMOTESampler(), classifier=RandomForestEstimator(), classifier__max_depth=None, classifier__n_estimators=100, imputer=IterativeModelImputer(), normalizer=RobustScalerNorm(), selector=TreeBasedSelection(), tamer=IsolationForestTamer(); total time=   0.1s
[CV] END balancer=SMOTESampler(), classifier=RandomForestEstimator(), classifier__max_depth=None, classifier__n_estimators=100, imputer=IterativeModelImputer(), normalizer=RobustScaler

KeyboardInterrupt: 

## Inspect the winner

In [ ]:
print(f'Best CV ROC-AUC : {search.best_score_:.4f}')
print('Best pipeline   :')
for name, step in search.best_estimator_.named_steps.items():
    label = 'passthrough' if isinstance(step, str) else type(step).__name__
    print(f'  {name:11s} -> {label}')

print('\nBest params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

y_pred  = search.predict(x_val)
y_proba = search.predict_proba(x_val)[:, 1]
val_auc = roc_auc_score(y_val, y_proba)

print(f'\nHold-out ROC-AUC : {val_auc:.4f}')
print('\nConfusion matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification report:')
print(classification_report(y_val, y_pred, zero_division=0))

## Leaderboard

In [ ]:
cv_df = (
    pd.DataFrame(search.cv_results_)
      .sort_values('mean_test_score', ascending=False)
      [['mean_test_score', 'std_test_score', 'params']]
      .head(10)
      .reset_index(drop=True)
)
cv_df

## Persist artifacts

In [ ]:
out_dir = os.path.dirname(path) or '.'
os.makedirs(out_dir, exist_ok=True)

joblib.dump(search.best_estimator_, os.path.join(out_dir, 'pipeline.pkl'))
joblib.dump(all_mappings,           os.path.join(out_dir, 'all_mapping.pkl'))
joblib.dump(y_mappings,             os.path.join(out_dir, 'y_mappings.pkl'))
joblib.dump(list(X.columns),        os.path.join(out_dir, 'features.pkl'))

print('Saved:', os.path.join(out_dir, 'pipeline.pkl'))